# MyGlico: validação end-to-end de rastreabilidade

Este notebook apresenta uma validação em formato Colab/Jupyter para o pipeline semântico do MyGlico.

A pergunta vem do slide **"Próximos Passos: Validação End-to-End"**:

> Uma medição consultada via SPARQL consegue comprovar, de ponta a ponta, que valor, horário, classe clínica e ponteiro para a linha exata do arquivo bruto UCI batem com o registro relacional no MySQL?

Nesta execução, validamos o `Paciente UCI 01` em `1991-04-29`, cruzando MySQL, GraphDB e arquivo bruto UCI.

## Resultado executivo

| Check | Resultado |
| --- | --- |
| Pacientes no MySQL | 70 |
| Pacientes no GraphDB | 70 |
| Medições no MySQL | 13.531 |
| Medições no GraphDB | 13.531 |
| Paciente validado | `Paciente UCI 01` |
| Data validada | `1991-04-29` |
| Linhas UCI validadas | `data-01:50`, `data-01:54`, `data-01:56`, `data-01:58` |
| Conclusão | MySQL e GraphDB batem por UUID, valor, horário, contexto, classe clínica e origem |

In [ ]:
# Em Colab, descomente se precisar instalar dependências.
# !pip install pymysql pandas requests

import pandas as pd
import pymysql
import requests

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3307,
    "user": "myglico",
    "password": "myglico",
    "database": "myglico",
}

GRAPHDB_ENDPOINT = "http://localhost:7200/repositories/class"
GRAPH_IRI = "https://w3id.org/myglico/ontology/mygv-dpo-extension"

## 1. Sanity check: MySQL e GraphDB têm o mesmo volume semântico?

Antes do tracking fino, validamos que as duas camadas representam o mesmo universo mínimo: pacientes e medições.

In [ ]:
mysql_counts_sql = """
SELECT 'patients' AS entity, COUNT(*) AS total FROM patients
UNION ALL
SELECT 'glucose_measurements' AS entity, COUNT(*) AS total FROM glucose_records;
"""

with pymysql.connect(**MYSQL_CONFIG) as conn:
    mysql_counts = pd.read_sql(mysql_counts_sql, conn)

mysql_counts

Resultado observado no MySQL:

| entity | total |
| --- | ---: |
| `patients` | 70 |
| `glucose_measurements` | 13.531 |

In [ ]:
sparql_counts = """
PREFIX mygv: <https://w3id.org/myglico/vocab/>

SELECT (COUNT(DISTINCT ?patient) AS ?totalPatients)
       (COUNT(DISTINCT ?measurement) AS ?totalMeasurements)
WHERE {
  GRAPH <https://w3id.org/myglico/ontology/mygv-dpo-extension> {
    ?patient a mygv:Patient .
    OPTIONAL { ?measurement a mygv:GlucoseMeasurement . }
  }
}
"""

response = requests.post(
    GRAPHDB_ENDPOINT,
    data={"query": sparql_counts},
    headers={"Accept": "application/sparql-results+json"},
    timeout=30,
)
response.raise_for_status()
response.json()["results"]["bindings"]

Resultado observado no GraphDB:

| totalPatients | totalMeasurements |
| ---: | ---: |
| 70 | 13.531 |

## 2. Visão relacional: MySQL como trilha de auditoria

A consulta abaixo parte de `glucose_records`, passa por `glucose_record_sources` e chega em `raw_import_records`, onde ficam `source_file`, `source_line` e o payload bruto.

In [ ]:
mysql_tracking_sql = """
SELECT
  p.name AS patient_name,
  psl.external_patient_code,
  rir.source_file,
  rir.source_line,
  CONCAT(rir.source_file, ':', rir.source_line) AS sourceRecordId,
  rir.raw_payload,
  gr.id AS glucose_record_id,
  rir.id AS raw_import_record_id,
  gr.measured_at,
  gr.glucose_mg_dl,
  gr.context_label,
  CASE
    WHEN gr.glucose_mg_dl < 70 THEN 'HypoglycemicGlucoseMeasurement'
    WHEN gr.glucose_mg_dl <= 180 THEN 'InTargetRangeGlucoseMeasurement'
    ELSE 'HyperglycemicGlucoseMeasurement'
  END AS clinical_class
FROM glucose_records gr
JOIN patients p ON p.id = gr.patient_id
JOIN patient_source_links psl ON psl.patient_id = p.id
JOIN glucose_record_sources grs ON grs.glucose_record_id = gr.id
JOIN raw_import_records rir ON rir.id = grs.raw_import_record_id
WHERE psl.external_patient_code = 'uci-01'
  AND DATE(gr.measured_at) = '1991-04-29'
ORDER BY gr.measured_at;
"""

with pymysql.connect(**MYSQL_CONFIG) as conn:
    mysql_tracking = pd.read_sql(mysql_tracking_sql, conn)

mysql_tracking

Resultado observado:

| sourceRecordId | raw_payload | glucose_record_id | raw_import_record_id | measured_at | glucose_mg_dl | context_label | clinical_class |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| `data-01:50` | `04-29-1991\t7:39\t58\t128` | `4fd99dd9-880e-4b5f-a40f-991558309cd5` | `a7a64c9b-f051-4fc7-832d-abf959a722ba` | `1991-04-29 07:39:00` | 128 | `pre_breakfast` | `InTargetRangeGlucoseMeasurement` |
| `data-01:54` | `04-29-1991\t13:38\t60\t192` | `586d2fba-a344-4a11-842f-76c912f14165` | `96bc5a2f-548d-41cc-bec7-012671d540ea` | `1991-04-29 13:38:00` | 192 | `pre_lunch` | `HyperglycemicGlucoseMeasurement` |
| `data-01:56` | `04-29-1991\t16:50\t62\t263` | `cfa591ae-a727-4c82-879f-117b9ce4161f` | `817235d6-5116-4428-9b89-91f83c11fc10` | `1991-04-29 16:50:00` | 263 | `pre_supper` | `HyperglycemicGlucoseMeasurement` |
| `data-01:58` | `04-29-1991\t22:28\t48\t081` | `19e711fa-658c-4bce-a88f-424240270bd5` | `246f8a6e-8ac0-4b83-8211-a1404f344f7e` | `1991-04-29 22:28:00` | 81 | `unspecified` | `InTargetRangeGlucoseMeasurement` |

## 3. Visão semântica: GraphDB como camada de consulta

A consulta SPARQL percorre o named graph materializado via R2RML. Ela parte do `sourceRecordId`, encontra o `RawRecord`, sobe para a `GlucoseMeasurement`, recupera paciente, horário, valor, contexto e classe clínica materializada.

In [ ]:
sparql_tracking = """
PREFIX mygv: <https://w3id.org/myglico/vocab/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?sourceRecordId ?rawRecord ?measurement ?patient ?label ?when ?value ?context ?clinicalClass
WHERE {
  GRAPH <https://w3id.org/myglico/ontology/mygv-dpo-extension> {
    VALUES ?sourceRecordId { "data-01:50" "data-01:54" "data-01:56" "data-01:58" }
    ?rawRecord a mygv:RawRecord ;
               mygv:sourceRecordId ?sourceRecordId .
    ?measurement a mygv:GlucoseMeasurement ;
                 mygv:derivedFromRawRecord ?rawRecord ;
                 mygv:belongsToPatient ?patient ;
                 mygv:measuredAt ?when ;
                 mygv:glucoseValueMgDl ?value .
    OPTIONAL { ?measurement mygv:measurementContext ?context . }
    OPTIONAL { ?patient rdfs:label ?label . }
    OPTIONAL {
      VALUES ?clinicalClass {
        mygv:HypoglycemicGlucoseMeasurement
        mygv:InTargetRangeGlucoseMeasurement
        mygv:HyperglycemicGlucoseMeasurement
      }
      ?measurement a ?clinicalClass .
    }
  }
}
ORDER BY ?sourceRecordId
"""

response = requests.post(
    GRAPHDB_ENDPOINT,
    data={"query": sparql_tracking},
    headers={"Accept": "application/sparql-results+json"},
    timeout=30,
)
response.raise_for_status()
rows = []
for binding in response.json()["results"]["bindings"]:
    rows.append({key: value["value"] for key, value in binding.items()})

sparql_tracking_df = pd.DataFrame(rows)
sparql_tracking_df

Resultado observado:

| sourceRecordId | measurement | rawRecord | patient | when | value | context | clinicalClass |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| `data-01:50` | `glucose-measurement/4fd99dd9-880e-4b5f-a40f-991558309cd5` | `raw-record/a7a64c9b-f051-4fc7-832d-abf959a722ba` | `patient/d83181c3-dda4-4f80-8a7f-0173027deef5` | `1991-04-29T07:39:00` | 128.0 | `pre_breakfast` | `InTargetRangeGlucoseMeasurement` |
| `data-01:54` | `glucose-measurement/586d2fba-a344-4a11-842f-76c912f14165` | `raw-record/96bc5a2f-548d-41cc-bec7-012671d540ea` | `patient/d83181c3-dda4-4f80-8a7f-0173027deef5` | `1991-04-29T13:38:00` | 192.0 | `pre_lunch` | `HyperglycemicGlucoseMeasurement` |
| `data-01:56` | `glucose-measurement/cfa591ae-a727-4c82-879f-117b9ce4161f` | `raw-record/817235d6-5116-4428-9b89-91f83c11fc10` | `patient/d83181c3-dda4-4f80-8a7f-0173027deef5` | `1991-04-29T16:50:00` | 263.0 | `pre_supper` | `HyperglycemicGlucoseMeasurement` |
| `data-01:58` | `glucose-measurement/19e711fa-658c-4bce-a88f-424240270bd5` | `raw-record/246f8a6e-8ac0-4b83-8211-a1404f344f7e` | `patient/d83181c3-dda4-4f80-8a7f-0173027deef5` | `1991-04-29T22:28:00` | 81.0 | `unspecified` | `InTargetRangeGlucoseMeasurement` |

## 4. Visão única: o que bate entre as camadas

| Linha UCI | Valor bruto | MySQL UUID | GraphDB UUID | Horário | Valor | Classe clínica | Status |
| --- | --- | --- | --- | --- | ---: | --- | --- |
| `data-01:50` | `04-29-1991 7:39 58 128` | `4fd99dd9-880e-4b5f-a40f-991558309cd5` | `4fd99dd9-880e-4b5f-a40f-991558309cd5` | `07:39:00` | 128 | `InTargetRangeGlucoseMeasurement` | Bate |
| `data-01:54` | `04-29-1991 13:38 60 192` | `586d2fba-a344-4a11-842f-76c912f14165` | `586d2fba-a344-4a11-842f-76c912f14165` | `13:38:00` | 192 | `HyperglycemicGlucoseMeasurement` | Bate |
| `data-01:56` | `04-29-1991 16:50 62 263` | `cfa591ae-a727-4c82-879f-117b9ce4161f` | `cfa591ae-a727-4c82-879f-117b9ce4161f` | `16:50:00` | 263 | `HyperglycemicGlucoseMeasurement` | Bate |
| `data-01:58` | `04-29-1991 22:28 48 081` | `19e711fa-658c-4bce-a88f-424240270bd5` | `19e711fa-658c-4bce-a88f-424240270bd5` | `22:28:00` | 81 | `InTargetRangeGlucoseMeasurement` | Bate |

## Conclusão

A validação end-to-end foi bem-sucedida.

A camada relacional preserva a trilha de auditoria (`source_file`, `source_line`, `raw_payload`, UUIDs e regra de normalização). A camada semântica materializada no GraphDB permite consultar a mesma medição por SPARQL, preservando UUID, horário com segundos, valor glicêmico, contexto, classe clínica e vínculo com o registro bruto.

Isso atende ao critério de sucesso dos próximos passos: a consulta semântica não retorna apenas dados clínicos, mas também mantêm um caminho auditável até a linha exata do dataset UCI.